
# Notebook 1 v3.1 — Audit Downloaded Artifact Pool + Dataset Structure

目标：从 `/mydata/doc2validate/data/downloaded_artifacts` 出发，识别已经下载并解压的文章目录，并 merge 每篇文章的 `dataset_structure.json`，初步筛选 curated benchmark candidates。

v3.1 修正点：

1. 你的 `dataset_structure.json` 实际结构是：
   ```json
   {
     "article_id": "...",
     "extractor": "dataset_structure",
     "status": "success",
     "result": {
       "organization": ...,
       "files": [...],
       "validation_targets": ...,
       "execution_relevant_notes": ...
     }
   }
   ```
   所以 v3.1 会先 unwrap 到 `result`。
2. 不把 `archive/` 当作失败；重点看 `extracted/` 里是否有真实文件和 tabular 文件。
3. 输出初步 candidate label，用于判断 72 个 extracted strict tabular candidates 中哪些最可能进入 curated benchmark。


In [1]:

from pathlib import Path
import json
from collections import Counter

import pandas as pd

# ===== 路径配置 =====
DATA_ROOT = Path("/mydata/doc2validate/data")
DOWNLOADED_ARTIFACTS_DIR = DATA_ROOT / "downloaded_artifacts"
STRUCTURED_DOCS_DIR = DATA_ROOT / "structured_docs"

RUN_ROOT = Path("/mydata/doc2validate/results/runs/scidata_4293")
ANALYSIS_DIR = RUN_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = ANALYSIS_DIR / "downloaded_artifact_pool_audit_v3_1_schema.csv"
OUTPUT_XLSX = ANALYSIS_DIR / "downloaded_artifact_pool_audit_v3_1_schema.xlsx"

print("Downloaded artifacts dir:", DOWNLOADED_ARTIFACTS_DIR)
print("Structured docs dir:", STRUCTURED_DOCS_DIR)
print("Downloaded artifacts exists:", DOWNLOADED_ARTIFACTS_DIR.exists())
print("Structured docs exists:", STRUCTURED_DOCS_DIR.exists())
print("Output CSV:", OUTPUT_CSV)
print("Output XLSX:", OUTPUT_XLSX)


Downloaded artifacts dir: /mydata/doc2validate/data/downloaded_artifacts
Structured docs dir: /mydata/doc2validate/data/structured_docs
Downloaded artifacts exists: True
Structured docs exists: True
Output CSV: /mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v3_1_schema.csv
Output XLSX: /mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v3_1_schema.xlsx


In [2]:

# ===== Constants =====

MANIFEST_NAME = "ARTIFACT_DOWNLOAD_MANIFEST.json"

TABULAR_SUFFIXES = {
    ".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"
}

STRICT_TABULAR_SUFFIXES = {
    ".csv", ".tsv", ".xlsx", ".xls", ".parquet"
}

ARCHIVE_SUFFIXES = {
    ".zip", ".tar", ".gz", ".tgz", ".tar.gz", ".7z", ".rar", ".bz2", ".xz"
}

KNOWN_PROVIDER_DIRS = {
    "github", "zenodo", "direct", "figshare", "osf", "generic", "dataverse"
}

TABULAR_FORMATS = {
    "csv", "tsv", "xlsx", "xls", "excel", "parquet", "json", "jsonl", "txt"
}

STRICT_TABULAR_FORMATS = {
    "csv", "tsv", "xlsx", "xls", "excel", "parquet"
}

PRIMARY_ROLE_KEYWORDS = {
    "primary", "primary_data", "main", "main_data", "raw", "raw_data"
}

DERIVED_ROLE_KEYWORDS = {
    "derived", "derived_data", "processed", "intermediate"
}

METADATA_ROLE_KEYWORDS = {
    "metadata", "data_dictionary", "codebook", "documentation"
}

ANNOTATION_ROLE_KEYWORDS = {
    "annotation", "annotations", "label", "labels"
}

SOFTWARE_ROLE_KEYWORDS = {
    "software", "code", "script", "runtime", "environment", "notebook"
}

NON_TABULAR_OR_ENV_FORMATS = {
    "binary", "image", "images", "hdf5", "h5", "netcdf", "nc",
    "database", "mongodb", "sql", "sqlite", "vasp", "mat", "matlab",
    "rdata", "rds"
}


In [3]:

# ===== General helper functions =====

def safe_load_json(path: Path):
    if not path or not path.exists():
        return None

    for enc in ["utf-8", "utf-8-sig"]:
        try:
            with path.open("r", encoding=enc) as f:
                return json.load(f)
        except Exception:
            pass

    try:
        return {"_load_error": path.read_text(errors="replace")[:500]}
    except Exception as e:
        return {"_load_error": str(e)}


def is_hidden_or_system(path: Path) -> bool:
    name = path.name
    return (
        name.startswith(".")
        or name == "__MACOSX"
        or name == ".DS_Store"
        or name == "__pycache__"
    )


def suffix_of(path: Path) -> str:
    name = path.name.lower()
    if name.endswith(".tar.gz"):
        return ".tar.gz"
    return path.suffix.lower()


def is_archive(path: Path) -> bool:
    return suffix_of(path) in ARCHIVE_SUFFIXES


def is_tabular_like(path: Path) -> bool:
    return suffix_of(path) in TABULAR_SUFFIXES


def is_strict_tabular(path: Path) -> bool:
    return suffix_of(path) in STRICT_TABULAR_SUFFIXES


def iter_real_files(article_dir: Path):
    for p in article_dir.rglob("*"):
        if is_hidden_or_system(p):
            continue
        if p.is_file() and p.name != MANIFEST_NAME:
            yield p


def top_level_entries_except_manifest(article_dir: Path):
    entries = []
    for p in article_dir.iterdir():
        if is_hidden_or_system(p):
            continue
        if p.name == MANIFEST_NAME:
            continue
        entries.append(p)
    return entries


def is_under_extracted(article_dir: Path, path: Path) -> bool:
    rel_parts = path.relative_to(article_dir).parts
    return "extracted" in rel_parts


def is_under_archive_dir(article_dir: Path, path: Path) -> bool:
    rel_parts = path.relative_to(article_dir).parts
    return "archive" in rel_parts


def compact_dict(d):
    if not d:
        return ""
    return json.dumps(d, ensure_ascii=False, sort_keys=True)


def compact_list(xs, max_items=20):
    if not xs:
        return ""

    xs = [str(x) for x in xs if str(x) != ""]
    if not xs:
        return ""

    if len(xs) > max_items:
        return "; ".join(xs[:max_items]) + f"; ... (+{len(xs) - max_items})"

    return "; ".join(xs)


def normalize_str(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    return str(x).strip()


def normalize_lower(x):
    return normalize_str(x).lower().strip()


def as_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


In [4]:

# ===== dataset_structure helpers =====

def find_dataset_structure_path(article_id: str):
    candidates = [
        STRUCTURED_DOCS_DIR / article_id / "dataset_structure.json",
        DATA_ROOT / "structured_docs" / article_id / "dataset_structure.json",
        DATA_ROOT / "dataset_structure" / article_id / "dataset_structure.json",
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


def unwrap_schema(schema):
    """
    Real extractor output often looks like:
    {
      "article_id": "...",
      "extractor": "dataset_structure",
      "status": "success",
      "result": {
        "dataset_identity": ...,
        "organization": ...,
        "files": [...],
        "validation_targets": ...,
        "execution_relevant_notes": ...
      }
    }
    """
    if not isinstance(schema, dict):
        return schema

    for key in ["result", "dataset_structure", "structure", "logical_schema", "schema"]:
        if isinstance(schema.get(key), dict):
            return schema[key]

    return schema


def get_files_from_schema(schema):
    schema = unwrap_schema(schema)

    if not isinstance(schema, dict):
        return []

    for key in ["files", "dataset_files", "logical_files", "data_files", "artifacts"]:
        val = schema.get(key)
        if isinstance(val, list):
            return val

    org = schema.get("organization")
    if isinstance(org, dict):
        for key in ["files", "data_files", "logical_files", "artifacts"]:
            val = org.get(key)
            if isinstance(val, list):
                return val

    return []


def get_validation_targets(schema):
    schema = unwrap_schema(schema)

    if not isinstance(schema, dict):
        return []

    val = schema.get("validation_targets", [])
    if isinstance(val, list):
        return val
    if val:
        return [val]
    return []


def get_execution_notes(schema):
    schema = unwrap_schema(schema)

    if not isinstance(schema, dict):
        return []

    possible_keys = [
        "execution_relevant_notes",
        "execution_notes",
        "runtime_notes",
        "validation_notes",
    ]

    notes = []
    for key in possible_keys:
        val = schema.get(key)
        if isinstance(val, list):
            notes.extend([normalize_str(v) for v in val if normalize_str(v)])
        elif isinstance(val, str) and val.strip():
            notes.append(val.strip())

    return notes


def get_context_summary(schema):
    schema = unwrap_schema(schema)
    if not isinstance(schema, dict):
        return ""
    val = schema.get("context_summary", "")
    if isinstance(val, str):
        return val
    if isinstance(val, dict):
        return json.dumps(val, ensure_ascii=False)[:1000]
    return normalize_str(val)


In [5]:

# ===== file-level schema helper functions =====

def file_role(f):
    if not isinstance(f, dict):
        return ""

    for key in ["role", "file_role", "schema_type", "type", "category"]:
        val = normalize_lower(f.get(key))
        if val:
            return val

    return ""


def file_format(f):
    if not isinstance(f, dict):
        return ""

    for key in ["format", "expected_format", "file_format", "type"]:
        val = normalize_lower(f.get(key))
        if val:
            return val.replace(".", "")

    return ""


def file_name_or_path(f):
    if not isinstance(f, dict):
        return ""

    vals = []
    for key in [
        "logical_name",
        "name",
        "file_name",
        "filename",
        "path",
        "relative_path",
        "file_pattern",
        "expected_path",
        "expected_file",
        "location",
    ]:
        val = normalize_str(f.get(key))
        if val:
            vals.append(val)

    return " | ".join(vals)


def file_has_path_or_pattern(f):
    if not isinstance(f, dict):
        return False

    for key in [
        "path",
        "relative_path",
        "file_pattern",
        "expected_path",
        "expected_file",
        "location",
        "directory",
    ]:
        if normalize_str(f.get(key)):
            return True

    return False


def file_columns(f):
    """
    Compatible with:
    columns: ["a", "b"]
    columns: [{"name": "...", "semantic": "..."}]
    expected_columns: [...]
    column_semantics: [...]
    schema: {"columns": [...]}
    """
    if not isinstance(f, dict):
        return []

    for key in [
        "columns",
        "expected_columns",
        "column_semantics",
        "fields",
        "variables",
    ]:
        val = f.get(key)

        if isinstance(val, list):
            return val

        if isinstance(val, dict):
            for subkey in ["columns", "fields", "variables"]:
                subval = val.get(subkey)
                if isinstance(subval, list):
                    return subval

    schema = f.get("schema")
    if isinstance(schema, dict):
        for subkey in ["columns", "fields", "variables"]:
            subval = schema.get(subkey)
            if isinstance(subval, list):
                return subval

    return []


def column_has_semantics(col):
    if isinstance(col, str):
        return False

    if not isinstance(col, dict):
        return False

    semantic_keys = [
        "semantic",
        "semantics",
        "semantic_role",
        "description",
        "meaning",
        "definition",
        "role",
        "unit",
        "units",
        "data_type",
        "datatype",
        "type",
        "expected_values",
        "allowed_values",
        "notes",
    ]

    return any(normalize_str(col.get(k)) for k in semantic_keys)


def file_has_column_semantics(f):
    cols = file_columns(f)
    if not cols:
        return False

    return any(column_has_semantics(c) for c in cols)


def file_column_count(f):
    return len(file_columns(f))


def is_primary_role(role: str):
    r = normalize_lower(role)
    return any(k in r for k in PRIMARY_ROLE_KEYWORDS)


def is_derived_role(role: str):
    r = normalize_lower(role)
    return any(k in r for k in DERIVED_ROLE_KEYWORDS)


def is_metadata_role(role: str):
    r = normalize_lower(role)
    return any(k in r for k in METADATA_ROLE_KEYWORDS)


def is_annotation_role(role: str):
    r = normalize_lower(role)
    return any(k in r for k in ANNOTATION_ROLE_KEYWORDS)


def is_software_role(role: str):
    r = normalize_lower(role)
    return any(k in r for k in SOFTWARE_ROLE_KEYWORDS)


def is_tabular_format(fmt: str):
    f = normalize_lower(fmt).replace(".", "")
    return f in TABULAR_FORMATS


def is_strict_tabular_format(fmt: str):
    f = normalize_lower(fmt).replace(".", "")
    return f in STRICT_TABULAR_FORMATS


def is_non_tabular_or_env_format(fmt: str):
    f = normalize_lower(fmt).replace(".", "")
    return f in NON_TABULAR_OR_ENV_FORMATS


In [6]:

# ===== schema feature extraction =====

def extract_schema_features(article_id: str):
    path = find_dataset_structure_path(article_id)

    if path is None:
        return {
            "has_dataset_structure": False,
            "dataset_structure_path": "",
        }

    raw_schema = safe_load_json(path)

    if not isinstance(raw_schema, dict):
        return {
            "has_dataset_structure": False,
            "dataset_structure_path": str(path),
            "dataset_structure_load_error": "not_a_dict_or_load_failed",
        }

    schema = unwrap_schema(raw_schema)

    if not isinstance(schema, dict):
        return {
            "has_dataset_structure": False,
            "dataset_structure_path": str(path),
            "dataset_structure_load_error": "unwrapped_schema_not_dict",
        }

    files = get_files_from_schema(raw_schema)
    validation_targets = get_validation_targets(raw_schema)
    execution_notes = get_execution_notes(raw_schema)

    roles = [file_role(f) for f in files if isinstance(f, dict)]
    formats = [file_format(f) for f in files if isinstance(f, dict)]
    formats = [f for f in formats if f]

    primary_files = [
        f for f in files
        if isinstance(f, dict) and is_primary_role(file_role(f))
    ]

    derived_files = [
        f for f in files
        if isinstance(f, dict) and is_derived_role(file_role(f))
    ]

    metadata_files = [
        f for f in files
        if isinstance(f, dict) and is_metadata_role(file_role(f))
    ]

    annotation_files = [
        f for f in files
        if isinstance(f, dict) and is_annotation_role(file_role(f))
    ]

    software_files = [
        f for f in files
        if isinstance(f, dict) and is_software_role(file_role(f))
    ]

    tabular_files = [
        f for f in files
        if isinstance(f, dict) and is_tabular_format(file_format(f))
    ]

    strict_tabular_files = [
        f for f in files
        if isinstance(f, dict) and is_strict_tabular_format(file_format(f))
    ]

    primary_tabular_files = [
        f for f in primary_files
        if is_tabular_format(file_format(f))
    ]

    primary_strict_tabular_files = [
        f for f in primary_files
        if is_strict_tabular_format(file_format(f))
    ]

    non_tabular_or_env_files = [
        f for f in files
        if isinstance(f, dict) and is_non_tabular_or_env_format(file_format(f))
    ]

    files_with_columns = [
        f for f in files
        if isinstance(f, dict) and len(file_columns(f)) > 0
    ]

    files_with_column_semantics = [
        f for f in files
        if isinstance(f, dict) and file_has_column_semantics(f)
    ]

    known_path_files = [
        f for f in files
        if isinstance(f, dict) and file_has_path_or_pattern(f)
    ]

    organization = schema.get("organization", "")
    if isinstance(organization, dict):
        organization_type = normalize_str(
            organization.get("type")
            or organization.get("organization_type")
            or organization.get("structure")
            or organization.get("description")
        )
    else:
        organization_type = normalize_str(organization)

    structure_confidence = schema.get("structure_confidence", schema.get("confidence", ""))

    role_counts = Counter([r for r in roles if r])
    format_counts = Counter([f for f in formats if f])

    primary_formats = [
        file_format(f)
        for f in primary_files
        if file_format(f)
    ]

    total_column_count = sum(file_column_count(f) for f in files if isinstance(f, dict))

    sample_logical_files = [
        file_name_or_path(f)
        for f in files[:20]
        if isinstance(f, dict)
    ]

    sample_primary_files = [
        file_name_or_path(f)
        for f in primary_files[:20]
        if isinstance(f, dict)
    ]

    sample_files_with_column_semantics = [
        file_name_or_path(f)
        for f in files_with_column_semantics[:20]
        if isinstance(f, dict)
    ]

    return {
        "has_dataset_structure": True,
        "dataset_structure_path": str(path),

        "organization_type": organization_type,
        "structure_confidence": structure_confidence,

        "logical_file_count": len(files),
        "primary_file_count": len(primary_files),
        "derived_file_count": len(derived_files),
        "metadata_file_count": len(metadata_files),
        "annotation_file_count": len(annotation_files),
        "software_file_count": len(software_files),

        "tabular_logical_file_count": len(tabular_files),
        "strict_tabular_logical_file_count": len(strict_tabular_files),
        "primary_tabular_file_count": len(primary_tabular_files),
        "primary_strict_tabular_file_count": len(primary_strict_tabular_files),
        "non_tabular_or_env_logical_file_count": len(non_tabular_or_env_files),

        "files_with_columns_count": len(files_with_columns),
        "files_with_column_semantics_count": len(files_with_column_semantics),
        "total_declared_column_count": total_column_count,
        "known_path_or_pattern_count": len(known_path_files),

        "has_primary_tabular": len(primary_tabular_files) > 0,
        "has_primary_strict_tabular": len(primary_strict_tabular_files) > 0,
        "has_column_semantics": len(files_with_column_semantics) > 0,
        "has_validation_targets": len(validation_targets) > 0,
        "has_execution_relevant_notes": len(execution_notes) > 0,

        "logical_formats": compact_list(sorted(set(formats))),
        "primary_formats": compact_list(sorted(set(primary_formats))),
        "role_counts": compact_dict(dict(role_counts)),
        "format_counts": compact_dict(dict(format_counts)),

        "validation_target_count": len(validation_targets),
        "execution_relevant_note_count": len(execution_notes),

        "context_summary_preview": get_context_summary(raw_schema)[:500],

        "sample_logical_files": compact_list(sample_logical_files, max_items=20),
        "sample_primary_files": compact_list(sample_primary_files, max_items=20),
        "sample_files_with_column_semantics": compact_list(sample_files_with_column_semantics, max_items=20),
    }


In [7]:

# ===== Optional sanity check for known examples =====

sample_ids = [
    "s41597-024-04232-w",
    "s41597-023-02060-y",
    "s41597-020-0407-9",
]

for article_id in sample_ids:
    path = find_dataset_structure_path(article_id)
    raw_schema = safe_load_json(path) if path else None
    schema = unwrap_schema(raw_schema)

    print("\n" + "=" * 80)
    print(article_id)
    print("path:", path)
    print("unwrapped keys:", list(schema.keys()) if isinstance(schema, dict) else None)

    files = get_files_from_schema(raw_schema)
    print("files len:", len(files))
    if files:
        print("first file keys:", list(files[0].keys()))
        print("first file preview:")
        print(json.dumps(files[0], ensure_ascii=False, indent=2)[:1200])



s41597-024-04232-w
path: /mydata/doc2validate/data/structured_docs/s41597-024-04232-w/dataset_structure.json
unwrapped keys: ['dataset_identity', 'organization', 'files', 'sample_or_record_unit', 'spatial_temporal_coverage', 'validation_targets', 'execution_relevant_notes', 'structure_confidence', 'context_summary']
files len: 8
first file keys: ['logical_name', 'file_pattern', 'format', 'schema_type', 'role', 'source', 'path', 'structure']
first file preview:
{
  "logical_name": "final_dataset",
  "file_pattern": "dataset_final.csv",
  "format": "csv",
  "schema_type": "tabular",
  "role": "primary_data",
  "source": "paper",
  "path": "outputs/dataset_final.csv",
  "structure": {
    "columns": {
      "name": {
        "data_type": "string",
        "semantic_type": "chemical_name",
        "description": "Common or IUPAC name of the chemical",
        "unit": null,
        "required": true
      },
      "CAS": {
        "data_type": "string",
        "semantic_type": "chemical_id

In [8]:

# ===== downloaded_artifacts scan =====

article_dirs = sorted([p for p in DOWNLOADED_ARTIFACTS_DIR.iterdir() if p.is_dir()])
print("Article dirs:", len(article_dirs))

rows = []

for article_dir in article_dirs:
    article_id = article_dir.name

    top_entries = top_level_entries_except_manifest(article_dir)
    top_dirs = [p.name for p in top_entries if p.is_dir()]
    top_files = [p.name for p in top_entries if p.is_file()]

    real_files = list(iter_real_files(article_dir))
    real_dirs = [
        p for p in article_dir.rglob("*")
        if p.is_dir() and not is_hidden_or_system(p)
    ]

    archive_files = [p for p in real_files if is_archive(p)]
    archive_dir_files = [p for p in real_files if is_under_archive_dir(article_dir, p)]

    extracted_files = [
        p for p in real_files
        if is_under_extracted(article_dir, p)
    ]

    non_archive_non_manifest_files = [
        p for p in real_files
        if not is_under_archive_dir(article_dir, p)
    ]

    extracted_tabular_files = [
        p for p in extracted_files
        if is_tabular_like(p)
    ]

    extracted_strict_tabular_files = [
        p for p in extracted_files
        if is_strict_tabular(p)
    ]

    non_archive_tabular_files = [
        p for p in non_archive_non_manifest_files
        if is_tabular_like(p)
    ]

    non_archive_strict_tabular_files = [
        p for p in non_archive_non_manifest_files
        if is_strict_tabular(p)
    ]

    file_suffix_counts = Counter(suffix_of(p) or "[no_suffix]" for p in real_files)
    extracted_suffix_counts = Counter(suffix_of(p) or "[no_suffix]" for p in extracted_files)
    non_archive_suffix_counts = Counter(
        suffix_of(p) or "[no_suffix]" for p in non_archive_non_manifest_files
    )

    provider_dirs_present = sorted([
        d for d in top_dirs
        if d.lower() in KNOWN_PROVIDER_DIRS
    ])

    if len(top_entries) == 0:
        artifact_presence = "manifest_only"
    elif len(real_files) == 0:
        artifact_presence = "dirs_but_no_files"
    else:
        artifact_presence = "has_real_artifact"

    has_extracted_content = len(extracted_files) > 0
    has_non_archive_content = len(non_archive_non_manifest_files) > 0
    has_archive_backup = len(archive_files) > 0 or len(archive_dir_files) > 0

    if artifact_presence != "has_real_artifact":
        downloaded_pool_status_v2 = "exclude_no_downloaded_content"

    elif len(extracted_strict_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_extracted_strict_tabular_candidate"

    elif len(extracted_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_extracted_tabular_like_candidate"

    elif len(non_archive_strict_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_non_archive_strict_tabular_candidate"

    elif len(non_archive_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_non_archive_tabular_like_candidate"

    elif has_extracted_content:
        downloaded_pool_status_v2 = "has_extracted_non_tabular_content"

    elif has_archive_backup and not has_non_archive_content:
        downloaded_pool_status_v2 = "archive_backup_only_no_extracted_content"

    else:
        downloaded_pool_status_v2 = "has_non_archive_non_tabular_content"

    row = {
        "article_id": article_id,
        "article_dir": str(article_dir),

        "artifact_presence": artifact_presence,
        "downloaded_pool_status_v2": downloaded_pool_status_v2,

        "top_level_dirs_except_manifest": compact_list(top_dirs),
        "top_level_files_except_manifest": compact_list(top_files),
        "provider_dirs_present": compact_list(provider_dirs_present),

        "has_archive_backup": has_archive_backup,
        "has_extracted_content": has_extracted_content,
        "has_non_archive_content": has_non_archive_content,

        "real_file_count_except_manifest": len(real_files),
        "real_dir_count": len(real_dirs),

        "archive_file_count": len(archive_files),
        "archive_dir_file_count": len(archive_dir_files),

        "extracted_file_count": len(extracted_files),
        "non_archive_file_count": len(non_archive_non_manifest_files),

        "extracted_tabular_like_file_count": len(extracted_tabular_files),
        "extracted_strict_tabular_file_count": len(extracted_strict_tabular_files),

        "non_archive_tabular_like_file_count": len(non_archive_tabular_files),
        "non_archive_strict_tabular_file_count": len(non_archive_strict_tabular_files),

        "file_suffix_counts_all": compact_dict(dict(file_suffix_counts)),
        "extracted_suffix_counts": compact_dict(dict(extracted_suffix_counts)),
        "non_archive_suffix_counts": compact_dict(dict(non_archive_suffix_counts)),

        "sample_extracted_files": compact_list(
            [str(p.relative_to(article_dir)) for p in extracted_files[:30]],
            max_items=30
        ),
        "sample_extracted_tabular_files": compact_list(
            [str(p.relative_to(article_dir)) for p in extracted_tabular_files[:20]],
            max_items=20
        ),
    }

    row.update(extract_schema_features(article_id))
    rows.append(row)

df = pd.DataFrame(rows)
df.head()


Article dirs: 187


,article_id,article_dir,artifact_presence,downloaded_pool_status_v2,top_level_dirs_except_manifest,top_level_files_except_manifest,provider_dirs_present,has_archive_backup,has_extracted_content,has_non_archive_content,...,logical_formats,primary_formats,role_counts,format_counts,validation_target_count,execution_relevant_note_count,context_summary_preview,sample_logical_files,sample_primary_files,sample_files_with_column_semantics
0,s41597-019-0021-x,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_non_archive_strict_tabular_candidate,github,,github,False,False,True,...,,,,,0,0,"{""requested_strategy"": ""section_focused_contex...",,,
1,s41597-019-0035-4,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_strict_tabular_candidate,github,,github,True,True,True,...,csv; json,json,"{""annotation"": 1, ""derived_data"": 3, ""metadata...","{""csv"": 4, ""json"": 2}",4,6,"{""requested_strategy"": ""section_focused_contex...",anatomical_iqms | https://figshare.com/article...,api_data_stream | https://mriqc.nimh.nih.gov/a...,
2,s41597-019-0098-2,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_strict_tabular_candidate,github,,github,True,True,True,...,binary; text; tsv; unknown,binary,"{""annotation"": 1, ""derived_data"": 4, ""metadata...","{""binary"": 4, ""text"": 1, ""tsv"": 1, ""unknown"": 2}",4,6,"{""requested_strategy"": ""section_focused_contex...",raw_data_tar_gz | ftp://ftp-trace.ncbi.nlm.nih...,raw_data_tar_gz | ftp://ftp-trace.ncbi.nlm.nih...,
3,s41597-019-0213-4,/mydata/doc2validate/data/downloaded_artifacts...,manifest_only,exclude_no_downloaded_content,,,,False,False,False,...,csv; json; php; sql; unknown,csv; unknown,"{""derived_data"": 1, ""metadata"": 2, ""primary_da...","{""csv"": 4, ""json"": 1, ""php"": 1, ""sql"": 1, ""unk...",4,6,"{""requested_strategy"": ""section_focused_contex...",donors | data/upload/donors.csv | donors.csv; ...,donors | data/upload/donors.csv | donors.csv; ...,
4,s41597-019-0342-9,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_tabular_like_candidate,github,,github,True,True,True,...,binary; text; unknown,text,"{""derived_data"": 1, ""metadata"": 1, ""primary_da...","{""binary"": 1, ""text"": 1, ""unknown"": 1}",2,5,"{""requested_strategy"": ""section_focused_contex...",pgxcorpus_annotations | PGxCorpus.tar | PGxCor...,pgxcorpus_annotations | PGxCorpus.tar | PGxCor...,


In [9]:

# ===== Candidate decision rules =====

def decide_candidate(row):
    has_real = row.get("artifact_presence") == "has_real_artifact"
    has_extracted_strict_tabular = row.get("extracted_strict_tabular_file_count", 0) > 0
    has_extracted_tabular = row.get("extracted_tabular_like_file_count", 0) > 0

    has_schema = bool(row.get("has_dataset_structure"))
    has_primary_tabular = bool(row.get("has_primary_tabular"))
    has_column_semantics = bool(row.get("has_column_semantics"))
    has_known_path = row.get("known_path_or_pattern_count", 0) > 0
    has_validation_targets = bool(row.get("has_validation_targets"))
    has_execution_notes = bool(row.get("has_execution_relevant_notes"))

    non_tabular_or_env_count = row.get("non_tabular_or_env_logical_file_count", 0)
    software_count = row.get("software_file_count", 0)

    if not has_real:
        return "exclude_no_downloaded_content"

    if not has_schema:
        if has_extracted_strict_tabular:
            return "maybe_schema_missing_but_tabular_downloaded"
        return "exclude_no_schema"

    # v3.1: high priority 不再强制要求 primary role，避免 role parser/LLM role label 不稳定
    if has_extracted_strict_tabular and has_column_semantics and has_known_path:
        return "yes_high_priority"

    if has_extracted_strict_tabular and has_primary_tabular and has_column_semantics:
        return "yes_after_path_manual_check"

    if has_extracted_strict_tabular and has_column_semantics:
        return "yes_after_role_or_path_manual_check"

    if has_extracted_tabular and has_column_semantics:
        return "yes_after_format_manual_check"

    if has_extracted_strict_tabular and has_schema:
        return "maybe_schema_or_column_semantics_missing"

    if non_tabular_or_env_count > 0 or software_count > 0:
        return "no_non_tabular_or_env_heavy"

    return "exclude_or_low_priority"


def decide_runtime_readiness(row):
    if row.get("artifact_presence") != "has_real_artifact":
        return "none_no_downloaded_content"

    if row.get("extracted_strict_tabular_file_count", 0) > 0:
        if row.get("non_tabular_or_env_logical_file_count", 0) > 0 or row.get("software_file_count", 0) > 0:
            return "medium_tabular_present_but_env_components"
        return "high_general_tabular_loader"

    if row.get("extracted_tabular_like_file_count", 0) > 0:
        return "medium_json_txt_or_non_strict_tabular"

    if row.get("non_tabular_or_env_logical_file_count", 0) > 0 or row.get("software_file_count", 0) > 0:
        return "low_specialized_runtime"

    return "unknown_or_low"


def candidate_reason(row):
    parts = []

    if row.get("extracted_strict_tabular_file_count", 0) > 0:
        parts.append("extracted_strict_tabular_present")
    elif row.get("extracted_tabular_like_file_count", 0) > 0:
        parts.append("extracted_tabular_like_present")
    else:
        parts.append("no_extracted_tabular")

    parts.append("has_schema" if row.get("has_dataset_structure") else "missing_schema")
    parts.append("primary_tabular" if row.get("has_primary_tabular") else "no_primary_tabular")
    parts.append("has_column_semantics" if row.get("has_column_semantics") else "no_column_semantics")
    parts.append("has_path_or_pattern" if row.get("known_path_or_pattern_count", 0) > 0 else "no_path_or_pattern")

    if row.get("has_validation_targets"):
        parts.append("has_validation_targets")

    if row.get("has_execution_relevant_notes"):
        parts.append("has_execution_notes")

    if row.get("non_tabular_or_env_logical_file_count", 0) > 0:
        parts.append("has_non_tabular_or_env_logical_files")

    if row.get("software_file_count", 0) > 0:
        parts.append("has_software_files")

    return "; ".join(parts)


df["candidate_for_curated_benchmark_v1"] = df.apply(decide_candidate, axis=1)
df["runtime_readiness_v1"] = df.apply(decide_runtime_readiness, axis=1)
df["candidate_reason_v1"] = df.apply(candidate_reason, axis=1)

df.head()


,article_id,article_dir,artifact_presence,downloaded_pool_status_v2,top_level_dirs_except_manifest,top_level_files_except_manifest,provider_dirs_present,has_archive_backup,has_extracted_content,has_non_archive_content,...,format_counts,validation_target_count,execution_relevant_note_count,context_summary_preview,sample_logical_files,sample_primary_files,sample_files_with_column_semantics,candidate_for_curated_benchmark_v1,runtime_readiness_v1,candidate_reason_v1
0,s41597-019-0021-x,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_non_archive_strict_tabular_candidate,github,,github,False,False,True,...,,0,0,"{""requested_strategy"": ""section_focused_contex...",,,,exclude_or_low_priority,unknown_or_low,no_extracted_tabular; has_schema; no_primary_t...
1,s41597-019-0035-4,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_strict_tabular_candidate,github,,github,True,True,True,...,"{""csv"": 4, ""json"": 2}",4,6,"{""requested_strategy"": ""section_focused_contex...",anatomical_iqms | https://figshare.com/article...,api_data_stream | https://mriqc.nimh.nih.gov/a...,,maybe_schema_or_column_semantics_missing,high_general_tabular_loader,extracted_strict_tabular_present; has_schema; ...
2,s41597-019-0098-2,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_strict_tabular_candidate,github,,github,True,True,True,...,"{""binary"": 4, ""text"": 1, ""tsv"": 1, ""unknown"": 2}",4,6,"{""requested_strategy"": ""section_focused_contex...",raw_data_tar_gz | ftp://ftp-trace.ncbi.nlm.nih...,raw_data_tar_gz | ftp://ftp-trace.ncbi.nlm.nih...,,maybe_schema_or_column_semantics_missing,medium_tabular_present_but_env_components,extracted_strict_tabular_present; has_schema; ...
3,s41597-019-0213-4,/mydata/doc2validate/data/downloaded_artifacts...,manifest_only,exclude_no_downloaded_content,,,,False,False,False,...,"{""csv"": 4, ""json"": 1, ""php"": 1, ""sql"": 1, ""unk...",4,6,"{""requested_strategy"": ""section_focused_contex...",donors | data/upload/donors.csv | donors.csv; ...,donors | data/upload/donors.csv | donors.csv; ...,,exclude_no_downloaded_content,none_no_downloaded_content,no_extracted_tabular; has_schema; primary_tabu...
4,s41597-019-0342-9,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_tabular_like_candidate,github,,github,True,True,True,...,"{""binary"": 1, ""text"": 1, ""unknown"": 1}",2,5,"{""requested_strategy"": ""section_focused_contex...",pgxcorpus_annotations | PGxCorpus.tar | PGxCor...,pgxcorpus_annotations | PGxCorpus.tar | PGxCor...,,no_non_tabular_or_env_heavy,medium_json_txt_or_non_strict_tabular,extracted_tabular_like_present; has_schema; no...


In [10]:

# ===== Summaries =====

print("Total article dirs:", len(df))
print("Has real artifact:", (df["artifact_presence"] == "has_real_artifact").sum())
print("Has extracted content:", df["has_extracted_content"].sum())
print("Has extracted strict tabular:", (df["extracted_strict_tabular_file_count"] > 0).sum())
print("Has dataset_structure:", df["has_dataset_structure"].sum())
print("Has column semantics:", df["has_column_semantics"].sum())
print("Has primary tabular:", df["has_primary_tabular"].sum())
print("Has path or pattern:", (df["known_path_or_pattern_count"] > 0).sum())

print("\nCandidate labels:")
display(df["candidate_for_curated_benchmark_v1"].value_counts(dropna=False).to_frame("count"))

print("\nRuntime readiness:")
display(df["runtime_readiness_v1"].value_counts(dropna=False).to_frame("count"))

print("\nHigh priority candidates:")
display(df[df["candidate_for_curated_benchmark_v1"] == "yes_high_priority"][[
    "article_id",
    "extracted_strict_tabular_file_count",
    "primary_tabular_file_count",
    "files_with_column_semantics_count",
    "known_path_or_pattern_count",
    "has_validation_targets",
    "has_execution_relevant_notes",
    "logical_formats",
    "primary_formats",
    "candidate_reason_v1",
]].head(100))


Total article dirs: 187
Has real artifact: 113
Has extracted content: 103
Has extracted strict tabular: 72
Has dataset_structure: 187
Has column semantics: 0
Has primary tabular: 98
Has path or pattern: 185

Candidate labels:


,count
exclude_no_downloaded_content,74
maybe_schema_or_column_semantics_missing,72
no_non_tabular_or_env_heavy,28
exclude_or_low_priority,13



Runtime readiness:


,count
none_no_downloaded_content,74
high_general_tabular_loader,41
medium_tabular_present_but_env_components,31
low_specialized_runtime,21
medium_json_txt_or_non_strict_tabular,13
unknown_or_low,7



High priority candidates:


,article_id,extracted_strict_tabular_file_count,primary_tabular_file_count,files_with_column_semantics_count,known_path_or_pattern_count,has_validation_targets,has_execution_relevant_notes,logical_formats,primary_formats,candidate_reason_v1


In [11]:

# ===== Useful subsets =====

df_high = df[df["candidate_for_curated_benchmark_v1"] == "yes_high_priority"].copy()

expandable_labels = [
    "yes_high_priority",
    "yes_after_path_manual_check",
    "yes_after_role_or_path_manual_check",
    "yes_after_format_manual_check",
    "maybe_schema_or_column_semantics_missing",
    "maybe_schema_missing_but_tabular_downloaded",
]

df_likely_expandable = df[
    df["candidate_for_curated_benchmark_v1"].isin(expandable_labels)
].copy()

df_downloaded_tabular = df[
    df["extracted_strict_tabular_file_count"] > 0
].copy()

print("High priority:", len(df_high))
print("Likely expandable:", len(df_likely_expandable))
print("Downloaded extracted strict tabular:", len(df_downloaded_tabular))

display(df_likely_expandable[[
    "article_id",
    "candidate_for_curated_benchmark_v1",
    "runtime_readiness_v1",
    "extracted_strict_tabular_file_count",
    "has_dataset_structure",
    "has_primary_tabular",
    "has_column_semantics",
    "known_path_or_pattern_count",
    "has_validation_targets",
    "has_execution_relevant_notes",
    "candidate_reason_v1",
]].head(150))


High priority: 0
Likely expandable: 72
Downloaded extracted strict tabular: 72


,article_id,candidate_for_curated_benchmark_v1,runtime_readiness_v1,extracted_strict_tabular_file_count,has_dataset_structure,has_primary_tabular,has_column_semantics,known_path_or_pattern_count,has_validation_targets,has_execution_relevant_notes,candidate_reason_v1
1,s41597-019-0035-4,maybe_schema_or_column_semantics_missing,high_general_tabular_loader,7,True,True,False,6,True,True,extracted_strict_tabular_present; has_schema; ...
2,s41597-019-0098-2,maybe_schema_or_column_semantics_missing,medium_tabular_present_but_env_components,3,True,False,False,8,True,True,extracted_strict_tabular_present; has_schema; ...
6,s41597-020-00609-9,maybe_schema_or_column_semantics_missing,high_general_tabular_loader,8,True,True,False,8,True,True,extracted_strict_tabular_present; has_schema; ...
7,s41597-020-00610-2,maybe_schema_or_column_semantics_missing,high_general_tabular_loader,4,True,True,False,4,True,True,extracted_strict_tabular_present; has_schema; ...
10,s41597-020-00676-y,maybe_schema_or_column_semantics_missing,medium_tabular_present_but_env_components,8,True,True,False,8,True,True,extracted_strict_tabular_present; has_schema; ...
...,...,...,...,...,...,...,...,...,...,...,...
160,s41597-025-05681-7,maybe_schema_or_column_semantics_missing,medium_tabular_present_but_env_components,2,True,False,False,8,True,True,extracted_strict_tabular_present; has_schema; ...
166,s41597-025-05889-7,maybe_schema_or_column_semantics_missing,medium_tabular_present_but_env_components,3,True,True,False,8,True,True,extracted_strict_tabular_present; has_schema; ...
181,sdata2018107,maybe_schema_or_column_semantics_missing,medium_tabular_present_but_env_components,14,True,False,False,6,True,True,extracted_strict_tabular_present; has_schema; ...
182,sdata2018141,maybe_schema_or_column_semantics_missing,medium_tabular_present_but_env_components,1,True,False,False,8,True,True,extracted_strict_tabular_present; has_schema; ...


In [12]:

# ===== Export =====

df.to_csv(OUTPUT_CSV, index=False)

df_high = df[df["candidate_for_curated_benchmark_v1"] == "yes_high_priority"].copy()

expandable_labels = [
    "yes_high_priority",
    "yes_after_path_manual_check",
    "yes_after_role_or_path_manual_check",
    "yes_after_format_manual_check",
    "maybe_schema_or_column_semantics_missing",
    "maybe_schema_missing_but_tabular_downloaded",
]

df_likely_expandable = df[
    df["candidate_for_curated_benchmark_v1"].isin(expandable_labels)
].copy()

df_downloaded_tabular = df[
    df["extracted_strict_tabular_file_count"] > 0
].copy()

df_no_content = df[df["artifact_presence"] != "has_real_artifact"].copy()

summary_candidate = df["candidate_for_curated_benchmark_v1"].value_counts(dropna=False)
summary_runtime = df["runtime_readiness_v1"].value_counts(dropna=False)
summary_download_status = df["downloaded_pool_status_v2"].value_counts(dropna=False)

try:
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="all")
        df_high.to_excel(writer, index=False, sheet_name="high_priority")
        df_likely_expandable.to_excel(writer, index=False, sheet_name="likely_expandable")
        df_downloaded_tabular.to_excel(writer, index=False, sheet_name="downloaded_tabular")
        df_no_content.to_excel(writer, index=False, sheet_name="no_real_artifact")
        summary_candidate.to_frame("count").to_excel(writer, sheet_name="summary_candidate")
        summary_runtime.to_frame("count").to_excel(writer, sheet_name="summary_runtime")
        summary_download_status.to_frame("count").to_excel(writer, sheet_name="summary_download")
    print("Saved Excel:", OUTPUT_XLSX)
except Exception as e:
    print("Excel export failed:", repr(e))
    print("CSV was still saved.")

print("Saved CSV:", OUTPUT_CSV)


Saved Excel: /mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v3_1_schema.xlsx
Saved CSV: /mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v3_1_schema.csv
